In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
ckey=pd.read_csv('cluster_keys.csv')
train_df=pd.read_csv('interpolations_filled_NaN_train.csv')
test_df=pd.read_csv('test.csv')


In [3]:
import json
with open ("full_parallel_hyp.json",'r') as f:
    high_compute_full_hyps=json.load(f)

In [4]:
#linear interpolated:
hyper_params_by_cluster={
2:
{'smoothing_level': 0.5, 'smoothing_slope': 0.375, 'trend': 'additive', 'seasonal': 'additive', 'seasonal_periods': 365},
1:
{'smoothing_level': 1.0, 'smoothing_slope': 0.0, 'trend': 'additive', 'seasonal': 'additive', 'seasonal_periods': 365},
5:
{'smoothing_level': 0.5, 'smoothing_slope': 0.0, 'trend': 'additive', 'seasonal': 'multiplicative', 'seasonal_periods': 365},
0:
{'smoothing_level': 0.25, 'smoothing_slope': 0.0, 'trend': 'multiplicative', 'seasonal': 'multiplicative', 'seasonal_periods': 365},
3:
{'smoothing_level': 0.0, 'smoothing_slope': 0.0, 'trend': 'additive', 'seasonal': 'multiplicative', 'seasonal_periods': 365},
4:
{'smoothing_level': 0.125, 'smoothing_slope': 0.0, 'trend': 'additive', 'seasonal': 'additive', 'seasonal_periods': 365}
}

In [5]:
from datetime import timedelta
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_percentage_error
from joblib import Parallel, delayed
import time

import warnings
warnings.filterwarnings('ignore')

class FFAExpoSmooth:
    def __init__(self,full_train,full_test,cluster_keys,variable_col:str='variable',target:str='num_sold',cluster_col:str='6_mean_clusters',cluster_params:dict=hyper_params_by_cluster,all_params:dict=high_compute_full_hyps):
        self.full_train=full_train
        self.full_test=full_test
        self.target=target
        self.variable_col=variable_col
        self.num_variables=self.full_train[self.variable_col].nunique()
        self.size=int(self.full_test.shape[0]/self.num_variables)
        self.cluster_col=cluster_col
        self.cluster_keys=dict(zip(cluster_keys[self.variable_col],cluster_keys[self.cluster_col]))
        self.cluster_params=cluster_params
        self.all_params=all_params
        self.predictions={}

    #remove yearly variance
    def take_recent_seasonal_shift(self):
        years_means={}
        for variable in self.full_train[self.variable_col].unique():
            years_means[variable]={}
            years_means[variable]['years']=[]
            years_means[variable]['means']={}
            for year in self.full_train.loc[self.full_train[self.variable_col]==variable,'year'].unique():
                years_means[variable]['years'].append(year)
                years_means[variable]['means'][year]=self.full_train.loc[(self.full_train[self.variable_col]==variable)&(self.full_train['year']==year),'num_sold'].mean()
            years_means[variable]['years']=sorted(years_means[variable]['years'])[::-1]
            diff=years_means[variable]['years'][0]-years_means[variable]['years'][1]
            for i in range(1,len(years_means[variable]['years'])-1):
                if i<=len(years_means[variable]['years'])-1:
                    finalized_year=years_means[variable]['years'][i]
                    year_to_process=years_means[variable]['years'][i+1]
                    curr_diff=self.full_train.loc[(self.full_train[self.variable_col]==variable)&(self.full_train['year']==finalized_year),'num_sold'].mean()-self.full_train.loc[(self.full_train[self.variable_col]==variable)&(self.full_train['year']==year_to_process),'num_sold'].mean()
                    self.full_train.loc[(self.full_train[self.variable_col]==variable)&(self.full_train['year']==year_to_process),'num_sold']+=(curr_diff-diff)
            years_means[variable]['new_means']={}
            for year in years_means[variable]['years']:
                new_mean=self.full_train.loc[(self.full_train[self.variable_col]==variable)&(self.full_train['year']==year),'num_sold'].mean()
                years_means[variable]['new_means'][year]=new_mean
            self.years_means=years_means

    def filter_data(self,funcdata5,splitting_variable:str):
        mask=funcdata5[self.variable_col]==splitting_variable
        funcdata5=funcdata5.loc[mask]
        funcdata5=funcdata5.drop_duplicates(subset='date',keep='first')
        funcdata5=funcdata5.sort_values(by='date')
        funcdata5=funcdata5.set_index('date')
        self.curr_subset=funcdata5    

    def split(self,filtered_timeseries):
        data = filtered_timeseries.copy()
        mn=data.index.min()
        mx=data.index.max()
        center = mx - timedelta(days=self.size)
        self.split_test = data.loc[center:]
        self.split_train = data.loc[:center - timedelta(days=1)]

    def iqr_outlier(self):
        for variable in self.full_train[self.variable_col].unique():
            q1,q3=np.percentile(self.full_train.loc[self.full_train[self.variable_col]==variable,self.target],[25,75])
            iqr=q3-q1
            upper=q3+(iqr*1.5)
            lower=q1-(iqr*1.5)
            if upper <=0: upper=1e-5
            if lower <=0: lower=1e-5
            self.full_train.loc[(self.full_train[self.variable_col]==variable)&(self.full_train[self.target]>upper),self.target]=upper
            self.full_train.loc[(self.full_train[self.variable_col]==variable)&(self.full_train[self.target]<lower),self.target]=lower
    
    def floor(self):
        for var in self.full_train[self.variable_col].unique():
            self.full_train.loc[(self.full_train[self.variable_col]==var)&(self.full_train[self.target]<=0),self.target]=1e-5

    def prep_test(self):
        self.full_test[self.variable_col]=self.full_test['country'].astype(str)+'\n'+self.full_test['store'].astype(str)+'\n'+self.full_test['product'].astype(str)
        self.full_test[self.target]=np.nan
        self.full_test['date']=pd.to_datetime(self.full_test['date'],format='%Y-%m-%d').dt.date
        self.full_test=self.full_test.sort_values(by='date')

    def gen_preds(self,subset_of_variable,all_custom:bool=False):
        curr_data=self.full_train.copy()
        curr_data=curr_data.loc[curr_data[self.variable_col]==subset_of_variable]
        if 'date' in curr_data.columns:
                curr_data=curr_data.sort_values(by='date')

        #data
        train=self.full_train.loc[self.full_train[self.variable_col]==subset_of_variable].iloc[-int((curr_data.shape[0]/90)-self.size):,:]
        test=self.full_test.loc[self.full_test[self.variable_col]==subset_of_variable]

        if not all_custom:
            #retrieve params based on cluster
            curr_cluster_key=self.cluster_keys.get(subset_of_variable,np.nan)
            if pd.isna(curr_cluster_key):
                print(f"{subset_of_variable.title()} has no cluster in keys")
                return None

            #params
            smoothing_level = self.cluster_params[curr_cluster_key]['smoothing_level']
            smoothing_slope = self.cluster_params[curr_cluster_key]['smoothing_slope']
            trend = self.cluster_params[curr_cluster_key]['trend']
            seasonal = self.cluster_params[curr_cluster_key]['seasonal']
            seasonal_periods = self.cluster_params[curr_cluster_key]['seasonal_periods']
        
        else:
            if subset_of_variable not in self.all_params:
                print(f"{subset_of_variable.title()} not in keys")
                return None
            elif pd.isna(self.all_params[subset_of_variable]):
                print(f"{subset_of_variable.title()} has no parameters")
                return None
            else:
                #params
                params=self.all_params.get(subset_of_variable,np.nan)
                smoothing_level = params.get('smoothing_level',np.nan)
                smoothing_slope = params.get('smoothing_slope',np.nan)
                trend = params.get('trend',np.nan)
                seasonal = params.get('seasonal',np.nan)
                seasonal_periods = params.get('seasonal_periods',np.nan)
        
        #build
        iter_model=ExponentialSmoothing(train[self.target],
                                        seasonal_periods=seasonal_periods,
                                        trend=trend,
                                        seasonal=seasonal)
        #fit
        iter_model_fit=iter_model.fit(smoothing_level=smoothing_level,
                                        smoothing_slope=smoothing_slope)
        #predict
        partition_test=test.copy()
        train.loc[train[self.target].isna(),self.target]=1e-5
        pred=iter_model_fit.forecast(test.shape[0])
        pred[pred==np.nan]=1e-5
        partition_test['forecast']=pred[:test.shape[0]].values
        if 'date' in partition_test.columns:
            partition_test=partition_test[['date',self.variable_col,'forecast']]
        else:
            partition_test=partition_test[[self.variable_col,'forecast']] 
        return (subset_of_variable,partition_test)

    def parallel_pred_calls(self,all_custom:bool=False):
        start = time.time()
        print(f'start: {start}')
        results=Parallel(n_jobs=-1, verbose=10)(delayed(self.gen_preds)(subset_of_variable,all_custom) for subset_of_variable in self.full_train[self.variable_col].unique())
        print("Elapsed:", time.time() - start)
        for tup in results:
            self.predictions[tup[0]]=tup[1]

    def fill_test(self):
        print('fill test')
        if not self.predictions:
            print("No predictions to fill.")
            return
        for k,v in self.predictions.items():
            if 'date' not in v.columns:
                v['date']=v.index
            v['date']=pd.to_datetime(v['date'],format='%Y-%m-%d').dt.date
            if (v['date'].values==self.full_test.loc[(self.full_test[self.variable_col]==k),'date'].values).all():
                self.full_test.loc[(self.full_test[self.variable_col]==k),self.target]=v['forecast'].values
            else:
                for date in v['date'].unique():
                    if self.full_test.loc[(self.full_test['date']==date)&(self.full_test[self.variable_col]==k),'date'].iloc[0]==v.loc[v['date']==date,'date'][0]:
                        self.full_test.loc[(self.full_test['date']==date)&(self.full_test[self.variable_col]==k),'date']=v.loc[v['date']==date,'date'][0]

In [6]:
ffa=FFAExpoSmooth(train_df,test_df,ckey)
       
ffa.take_recent_seasonal_shift()
 
ffa.iqr_outlier()
ffa.floor()

ffa.prep_test()

ffa.parallel_pred_calls()

#ffa.gen_preds("Italy\nPremium Sticker Mart\nKerneler")

ffa.fill_test()

#ffa.predictions

ffa.full_test.sample()

start: 1758718085.332067


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   10.0s
[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:   13.5s
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   14.9s
[Parallel(n_jobs=-1)]: Done  53 tasks      | elapsed:   17.0s
[Parallel(n_jobs=-1)]: Done  69 out of  90 | elapsed:   20.1s remaining:    6.0s
[Parallel(n_jobs=-1)]: Done  79 out of  90 | elapsed:   22.4s remaining:    3.0s
[Parallel(n_jobs=-1)]: Done  90 out of  90 | elapsed:   24.5s finished


Elapsed: 24.76443576812744
fill test


,id,date,country,store,product,variable,num_sold
41098,271228,2018-04-02,Kenya,Premium Sticker Mart,Kerneler,Kenya\nPremium Sticker Mart\nKerneler,22.706363


In [7]:
ffa.full_test.isna().sum()

id          0
date        0
country     0
store       0
product     0
variable    0
num_sold    0
dtype: int64

In [8]:
ffa.full_test.sample()

,id,date,country,store,product,variable,num_sold
27381,257511,2017-11-01,Finland,Stickers for Less,Kaggle,Finland\nStickers for Less\nKaggle,20097.142315


In [9]:
low_compute_exp_smth_submission=ffa.full_test[['id','num_sold']].sort_values(by='id')
low_compute_exp_smth_submission['num_sold']=low_compute_exp_smth_submission['num_sold'].astype(int)
low_compute_exp_smth_submission.sample()


,id,num_sold
2539,232669,383


In [10]:
low_compute_exp_smth_submission.to_csv('low_compute_submission.csv',index=False)